In [11]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pyreadr import read_r
from tqdm import tqdm
from tqdm.contrib.concurrent import process_map
import re
from scipy.stats import pearsonr, spearmanr


In [2]:
def find_matching_strings(A, B):
    """
    Find all strings in list A that match any pattern in list B.
    In list B, a '/' could separate two patterns and 'x' is a wildcard character that matches any single character.
    If a string matches multiple patterns, it will be included only once in the result.
    Args:
        A (list): List of strings to search through
        B (list): List of single or double patterns where 'x' is a wildcard

    Returns:
        list: All strings from A that match at least one pattern in B
    """
    # Convert patterns in B to regex patterns (replace 'x' with '.')
    regex_patterns = []
    for pattern in B:
        # a pattern can contain '/', which means 'or'
        sub_patterns = pattern.split('/')
        regex_sub_patterns = []
        for sub_pattern in sub_patterns:
            # Escape special regex characters, then replace 'x' with '.'
            escaped_sub_pattern = re.escape(sub_pattern).replace('x', '.')
            regex_sub_patterns.append(escaped_sub_pattern)
        
        # Join sub-patterns with '|' for an OR match, and anchor the whole expression
        full_regex_pattern = f"^({'|'.join(regex_sub_patterns)})$"
        regex_patterns.append(re.compile(full_regex_pattern))
    
    matching_strings = []
    matching_map = {}
    for string in A:
        # Check if the string matches any pattern in B
        for regex_pattern, b_pattern in zip(regex_patterns, B):
            if regex_pattern.match(string):
                matching_strings.append(string)
                matching_map[string] = b_pattern
                break  # Found a match, no need to check other patterns

    return matching_strings, matching_map

In [3]:
packer_s8 = pd.read_csv("data/aax1971_Table_S8.gz", sep="\t", index_col=0, compression="gzip")
packer_s8.sample(5)

,gene.id,lineage,raw.tpm.estimate,adjusted.tpm.estimate,bootstrap.median.tpm,ci.95p.lb,ci.80p.lb,ci.80p.ub,ci.95p.ub
gene,,,,,,,,,
zip-2,WBGene00019327,ABpxpapap,159.2,150.3,154.5,87.0,109.8,210.3,242.7
T13H5.8,WBGene00011761,Dxapa,52.2,44.0,51.1,16.7,27.9,76.6,91.8
Y55F3AR.1,WBGene00021932,ABalapxppap/ABalaappppp/ABalapaappp/ABalppappp...,13.0,8.4,12.2,0.0,2.4,24.3,31.1
F59C12.3,WBGene00019102,MSxpppap,120.7,111.1,119.6,52.3,75.2,168.4,199.4
srh-82,WBGene00005303,Cxp,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
packer_exp = read_r("./data/viscello/lin_sc_expr_190602.rds")[None]
protein_exp = pd.read_csv("data/protein/aggregated_all/s3.csv", index_col=0).T
packer_exp = packer_exp.fillna(0)
protein_exp = protein_exp.fillna(0)

# tf filtering for packer
tf_df = pd.read_csv("./data/full_tf_names.tsv", sep="\t")
tf_names = tf_df[~tf_df["Public name"].isna()]["Public name"].values
tf_names_alt = tf_df["Sequence name"].values
packer_tf_names = []
for name in packer_exp.index:
    if name in tf_names_alt or name in tf_names:
        packer_tf_names.append(name)
packer_exp = packer_exp.loc[packer_tf_names, :]

# get rid of empty rows and columns
packer_exp = packer_exp.loc[:, (packer_exp != 0).any(axis=0)]
packer_exp = packer_exp.loc[(packer_exp != 0).any(axis=1), :]
protein_exp = protein_exp.loc[:, (protein_exp != 0).any(axis=0)]
protein_exp = protein_exp.loc[(protein_exp != 0).any(axis=1), :]

In [5]:
matching_strings, matching_map = find_matching_strings(packer_exp.columns, packer_s8["lineage"].unique())

In [6]:
packer_exp.index

Index(['R02D3.7', 'nhr-76', 'Y55F3AM.14', 'nhr-41', 'nhr-122', 'nhr-92',
       'nhr-87', 'nhr-146', 'Y41D4B.26', 'nhr-274',
       ...
       'nhr-234', 'F58G1.2', 'moe-3', 'Y48C3A.12', 'efl-2', 'zip-3', 'hmg-1.1',
       'nhr-61', 'cog-1', 'Y53F4B.3'],
      dtype='object', length=663)

In [7]:
packer_s8.loc["R02D3.7"]

,gene.id,lineage,raw.tpm.estimate,adjusted.tpm.estimate,bootstrap.median.tpm,ci.95p.lb,ci.80p.lb,ci.80p.ub,ci.95p.ub
gene,,,,,,,,,
R02D3.7,WBGene00019824,28_cell_or_earlier,119.2,97.1,115.3,41.9,65.9,182.0,222.6
R02D3.7,WBGene00019824,ABalaaaa,30.8,19.7,29.0,0.0,9.0,54.4,72.5
R02D3.7,WBGene00019824,ABalaaaal,24.9,9.3,24.5,0.0,0.0,48.4,70.3
R02D3.7,WBGene00019824,ABalaaaala/ABalaapaaa,20.0,11.5,19.0,0.0,2.5,39.6,53.5
R02D3.7,WBGene00019824,ABalaaaalal/ABalaapaaar,112.8,56.4,107.2,0.0,0.0,228.8,280.7
...,...,...,...,...,...,...,...,...,...
R02D3.7,WBGene00019824,MSxppppp,50.8,45.1,48.7,14.9,25.6,78.3,90.8
R02D3.7,WBGene00019824,MSxppppx,26.7,10.4,26.5,0.0,0.0,58.0,75.7
R02D3.7,WBGene00019824,Z2/Z3:pseudotime_bin_1,274.1,285.8,273.2,226.6,241.8,308.2,324.4


In [8]:
packer_s8_by_gene = {}
for gene_name in tqdm(packer_exp.index):
    cur_packer_s8_exp = packer_s8.loc[gene_name]
    if not cur_packer_s8_exp.empty:
        packer_s8_by_gene[gene_name] = cur_packer_s8_exp

100%|██████████| 663/663 [01:15<00:00,  8.79it/s]


In [9]:
# comparision of packer and packer_s8
packer_exp_list = []
packer_s8_exp_list = []
packer_s8_exp_bootstrap_list = []
for packer_name, s8_name in tqdm(matching_map.items()):
    for gene_name, exp_array in packer_s8_by_gene.items():
        if s8_name in exp_array["lineage"].values:
            cur_packer_s8_exp = exp_array[exp_array["lineage"] == s8_name]
            if cur_packer_s8_exp.empty:
                continue
            if packer_exp.loc[gene_name, packer_name] == 0:
                continue
            packer_exp_list.append(packer_exp.loc[gene_name, packer_name])
            packer_s8_exp_list.append(cur_packer_s8_exp["adjusted.tpm.estimate"].values[0])
            packer_s8_exp_bootstrap_list.append(cur_packer_s8_exp["bootstrap.median.tpm"].values[0])

100%|██████████| 1010/1010 [02:26<00:00,  6.92it/s]


In [12]:
r, p = pearsonr(packer_s8_exp_bootstrap_list, packer_s8_exp_list)
print(f"Pearson correlation: {r}, p-value: {p}")


Pearson correlation: 0.996751635625425, p-value: 0.0


In [13]:
len(packer_exp_list), len(packer_s8_exp_list), len(packer_s8_exp_bootstrap_list)

(333237, 333237, 333237)

In [14]:
common_lineages = list(set(packer_exp.columns) & set(protein_exp.columns))
protein_exp = protein_exp[common_lineages]
protein_exp = protein_exp.loc[(protein_exp != 0).any(axis=1), :]
packer_exp = packer_exp[common_lineages]
packer_exp = packer_exp.loc[(packer_exp != 0).any(axis=1), :]


In [15]:
# record the l2 distances between each pair of common lineages
# across two different datasets, record the distances in two lists
packer_distances = []
protein_distances = []
common_lineages = list(set(packer_exp.columns) & set(protein_exp.columns))
common_packer_exp_values = packer_exp.values.T
common_protein_exp_values = protein_exp.values.T
all_pairs = [(i, j) for i in range(len(common_lineages)) for j in range(i + 1, len(common_lineages))]
def pair_distance(pair: tuple):
    i, j = pair
    return np.linalg.norm(common_packer_exp_values[i] - common_packer_exp_values[j]), \
           np.linalg.norm(common_protein_exp_values[i] - common_protein_exp_values[j])

packer_distances, protein_distances = zip(*process_map(
    pair_distance, 
    all_pairs, 
    max_workers=10, 
    chunksize=100, 
    desc="Calculating distances"
    ))

Calculating distances:   0%|          | 0/456490 [00:00<?, ?it/s]

In [16]:
# pearson r correlation between packer and protein distances
packer_distances = np.array(packer_distances)
protein_distances = np.array(protein_distances)
correlation, p = spearmanr(packer_distances, protein_distances)
print(f"Correlation between packer and protein distances: {correlation:.4f}, p-value: {p:.4e}")

Correlation between packer and protein distances: 0.1824, p-value: 0.0000e+00


In [18]:
# convert all packer indexes to capital case

packer_exp.index = [name.upper() for name in packer_exp.index]
protein_list = []
packer_list = []
for idx_name in protein_exp.index:
    tf_name = idx_name.split("_")[0]
    protein_vals = protein_exp.loc[idx_name, common_lineages]
    if tf_name in packer_exp.index:
        packer_vals = packer_exp.loc[tf_name, common_lineages]
        for protein_val, packer_val in zip(protein_vals, packer_vals):
            if protein_val != 0 and packer_val != 0:
                protein_list.append(protein_val)
                packer_list.append(packer_val)


correlation, p = spearmanr(packer_list, protein_list)
print(f"Correlation between packer and protein expression: {correlation:.4f}, p-value: {p:.4e}")

Correlation between packer and protein expression: 0.2431, p-value: 0.0000e+00


In [56]:
len(packer_list)

23576